# PBMC Cross-Site Transfer Diagnostics

Assess whether directional differences between Cambridge and Newcastle transfer persist after matching training cell-type support or donor counts.

This notebook generates the numerical summaries used in **Figure S4**. It reuses saved matching results when available; set the corresponding `RUN_*` flag to `True` to rerun an analysis. Classifier fitting requires the frozen PBMC benchmark-ready AnnData object.

Final figure assembly is handled by `notebooks/figures/supplementary_figures.ipynb`.

In [1]:
from pathlib import Path
import sys

import anndata as ad
import numpy as np
import pandas as pd
from IPython.display import display


REPO_ROOT = next(
    (
        p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (p / "src/scrna_benchmark").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from inside the repository.")

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from scrna_benchmark.diagnostics import (
    run_support_matched_group_transfer,
    run_donor_count_matched_group_transfer,
    summarize_transfer_sensitivity,
)
from scrna_benchmark.representations import validate_representations


ADATA_PATH = (
    REPO_ROOT / "data/PBMC_Stephenson/stephenson_benchmark_ready.h5ad"
)
TRANSFER_DIR = (
    REPO_ROOT / "results/pbmc_no_batch/group_transfer/Cambridge_vs_Ncl"
)
DIAG_DIR = TRANSFER_DIR / "diagnostics"
ORIGINAL_PATH = TRANSFER_DIR / "group_transfer_summary.csv"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

if not ORIGINAL_PATH.exists():
    raise FileNotFoundError(
        f"Run the original PBMC transfer benchmark first: {ORIGINAL_PATH}"
    )

REPRESENTATIONS = {
    "hvg": "hvg",
    "pca": "X_pca",
    "harmony": "X_harmony",
    "scvi": "X_scVI",
}
REP_ORDER = list(REPRESENTATIONS)
DIRECTIONS = ["Cambridge_to_Ncl", "Ncl_to_Cambridge"]

N_REPEATS = 10
RUN_SUPPORT_MATCH = False
RUN_DONOR_MATCH = False

# Load expression data only when classifiers must be rerun.
_pbmc = None


def get_pbmc():
    global _pbmc

    if _pbmc is None:
        if not ADATA_PATH.exists():
            raise FileNotFoundError(ADATA_PATH)

        candidate = ad.read_h5ad(ADATA_PATH)

        required = {"Site", "cell_type", "patient_id"}
        missing = required.difference(candidate.obs.columns)
        if missing:
            raise KeyError(f"Missing PBMC metadata columns: {sorted(missing)}")

        if not candidate.obs_names.is_unique:
            raise ValueError("PBMC cell identifiers must be unique.")

        observed = (
            candidate.n_obs,
            candidate.n_vars,
            candidate.obs["patient_id"].nunique(),
            candidate.obs["cell_type"].nunique(),
        )
        if observed != (10107, 1000, 23, 13):
            raise ValueError(f"Unexpected PBMC benchmark dimensions: {observed}")

        validate_representations(candidate, REPRESENTATIONS)
        _pbmc = candidate

    return _pbmc


def checked_summary(raw):
    expected = pd.MultiIndex.from_product(
        [REP_ORDER, DIRECTIONS],
        names=["representation", "direction"],
    )
    counts = raw.groupby(["representation", "direction"]).size()

    if (
        len(counts) != len(expected)
        or not counts.reindex(expected, fill_value=0).eq(N_REPEATS).all()
        or raw.duplicated(["representation", "direction", "repeat"]).any()
    ):
        raise ValueError(
            "Expected 10 distinct repeats for every representation and direction."
        )

    if not np.isfinite(raw["macro_f1"]).all():
        raise ValueError("Matching results contain invalid macro F1 values.")

    return summarize_transfer_sensitivity(raw)

In [2]:
SUPPORT_RAW = DIAG_DIR / "support_matched_transfer_raw.csv"
SUPPORT_SUMMARY = DIAG_DIR / "support_matched_transfer_summary.csv"

if RUN_SUPPORT_MATCH or not SUPPORT_RAW.exists():
    support_raw, support_sampling = run_support_matched_group_transfer(
        get_pbmc(),
        REPRESENTATIONS,
        group_col="Site",
        group_a="Cambridge",
        group_b="Ncl",
        celltype_col="cell_type",
        n_repeats=N_REPEATS,
        seed=5000,
    )
    support_summary = checked_summary(support_raw)

    support_raw.to_csv(SUPPORT_RAW, index=False)
    support_sampling.to_csv(
        DIAG_DIR / "support_matched_training_counts.csv",
        index=False,
    )
else:
    support_raw = pd.read_csv(SUPPORT_RAW)
    support_summary = checked_summary(support_raw)

support_summary.to_csv(SUPPORT_SUMMARY, index=False)
display(support_summary)

,analysis,representation,train_group,test_group,direction,macro_f1_mean,macro_f1_std,accuracy_mean,accuracy_std,n_runs
0,support_matched,harmony,Cambridge,Ncl,Cambridge_to_Ncl,0.734606,0.004108,0.820208,0.000911,10
1,support_matched,harmony,Ncl,Cambridge,Ncl_to_Cambridge,0.681605,0.002547,0.684670,0.001656,10
2,support_matched,hvg,Cambridge,Ncl,Cambridge_to_Ncl,0.716180,0.006864,0.804148,0.005879,10
3,support_matched,hvg,Ncl,Cambridge,Ncl_to_Cambridge,0.653025,0.012867,0.649016,0.020136,10
4,support_matched,pca,Cambridge,Ncl,Cambridge_to_Ncl,0.726147,0.004371,0.824489,0.003049,10
5,support_matched,pca,Ncl,Cambridge,Ncl_to_Cambridge,0.694154,0.006919,0.699109,0.005728,10
6,support_matched,scvi,Cambridge,Ncl,Cambridge_to_Ncl,0.707664,0.003285,0.809886,0.001450,10
7,support_matched,scvi,Ncl,Cambridge,Ncl_to_Cambridge,0.659155,0.002649,0.650963,0.001509,10


In [3]:
DONOR_RAW = DIAG_DIR / "donor_count_matched_transfer_raw.csv"
DONOR_SUMMARY = DIAG_DIR / "donor_count_matched_transfer_summary.csv"

if RUN_DONOR_MATCH or not DONOR_RAW.exists():
    donor_raw = run_donor_count_matched_group_transfer(
        get_pbmc(),
        REPRESENTATIONS,
        group_col="Site",
        group_a="Cambridge",
        group_b="Ncl",
        celltype_col="cell_type",
        donor_col="patient_id",
        n_repeats=N_REPEATS,
        seed=6000,
    )
    donor_summary = checked_summary(donor_raw)
    donor_raw.to_csv(DONOR_RAW, index=False)
else:
    donor_raw = pd.read_csv(DONOR_RAW)
    donor_summary = checked_summary(donor_raw)

donor_summary.to_csv(DONOR_SUMMARY, index=False)
display(donor_summary)

,analysis,representation,train_group,test_group,direction,macro_f1_mean,macro_f1_std,accuracy_mean,accuracy_std,n_runs
0,donor_count_matched,harmony,Cambridge,Ncl,Cambridge_to_Ncl,0.712381,0.002640,0.785907,0.004931,10
1,donor_count_matched,harmony,Ncl,Cambridge,Ncl_to_Cambridge,0.677627,0.004195,0.679511,0.004559,10
2,donor_count_matched,hvg,Cambridge,Ncl,Cambridge_to_Ncl,0.708164,0.002280,0.781786,0.003913,10
3,donor_count_matched,hvg,Ncl,Cambridge,Ncl_to_Cambridge,0.656836,0.014466,0.654837,0.018715,10
4,donor_count_matched,pca,Cambridge,Ncl,Cambridge_to_Ncl,0.740478,0.003184,0.819747,0.005268,10
5,donor_count_matched,pca,Ncl,Cambridge,Ncl_to_Cambridge,0.688731,0.007282,0.692045,0.008936,10
6,donor_count_matched,scvi,Cambridge,Ncl,Cambridge_to_Ncl,0.695861,0.002823,0.772083,0.004834,10
7,donor_count_matched,scvi,Ncl,Cambridge,Ncl_to_Cambridge,0.652814,0.001665,0.646634,0.001513,10


In [4]:
original = pd.read_csv(ORIGINAL_PATH)

tables = [
    ("original", original, "macro_f1"),
    ("support matched", support_summary, "macro_f1_mean"),
    ("donor-count matched", donor_summary, "macro_f1_mean"),
]

comparison_parts = []

for analysis, table, value_col in tables:
    wide = (
        table.pivot(
            index="representation",
            columns="direction",
            values=value_col,
        )
        .reindex(index=REP_ORDER, columns=DIRECTIONS)
    )

    if wide.isna().any().any():
        raise ValueError(f"Incomplete transfer results for {analysis}.")

    comparison_parts.append(
        pd.DataFrame({
            "analysis": analysis,
            "representation": REP_ORDER,
            "direction_gap_Cambridge_to_Ncl_minus_reverse": (
                wide["Cambridge_to_Ncl"] - wide["Ncl_to_Cambridge"]
            ).to_numpy(),
        })
    )

comparison = pd.concat(comparison_parts, ignore_index=True)
comparison.to_csv(
    DIAG_DIR / "transfer_direction_gap_comparison.csv",
    index=False,
)

display(comparison)
print("Transfer summaries and directional-gap table are ready for Figure S4.")

,analysis,representation,direction_gap_Cambridge_to_Ncl_minus_reverse
0,original,hvg,0.051382
1,original,pca,0.052195
2,original,harmony,0.036591
3,original,scvi,0.041500
4,support matched,hvg,0.063155
5,support matched,pca,0.031993
6,support matched,harmony,0.053001
7,support matched,scvi,0.048510
8,donor-count matched,hvg,0.051328
9,donor-count matched,pca,0.051747


Transfer summaries and directional-gap table are ready for Figure S4.
